# Download and prepare NCI THREDDS wind_nowcast pq files

This notebook downloads the NCI THREDDS `wind_nowcast` `.pq` files, keeps the original site-level folder structure, merges each site's monthly files into one site-level `.pq` file, and previews the merged Bellambi dataset.

Expected workflow:

1. Set paths and THREDDS constants.
2. Discover `.pq` files from the remote THREDDS catalog.
3. Download missing or changed files into `data/wind_nowcast`.
4. Merge files by site into `data/wind_nowcast/merged_by_site`.
5. Preview the merged Bellambi file.

## Setup

This cell imports dependencies, defines the THREDDS catalog URL, locates the project root, and creates the local output folder. The code is written so the notebook can be run from either the repository root or the notebook directory.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
import time
import xml.etree.ElementTree as ET

import requests

# tqdm is optional. If it is unavailable, downloads still work without progress bars.
try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# THREDDS exposes a machine-readable catalog.xml next to the catalog.html page.
CATALOG_URL = "https://thredds.nci.org.au/thredds/catalog/fx31/publications/wind_nowcast/catalog.xml"
THREDDS_ROOT = "https://thredds.nci.org.au"
CHUNK_SIZE = 1024 * 1024
TIMEOUT = 60

# XML namespaces used by THREDDS catalog documents.
NS = {"thredds": "http://www.unidata.ucar.edu/namespaces/thredds/InvCatalog/v1.0"}
XLINK = "{http://www.w3.org/1999/xlink}href"


def find_project_root(start=None):
    """Find the BOM-Team repo root from the current notebook working directory."""
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists() or path.name == "BOM-Team":
            return path
    raise RuntimeError(f"Could not find project root from {current}")


# Store downloaded files under the repository data directory.
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "wind_nowcast"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATA_DIR

(PosixPath('/home/timekeeper/Documents/Development/BOM-Team'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast'))

## Discover pq files from THREDDS

The root catalog contains eight site folders as `catalogRef` entries. This section opens each site catalog, extracts only files ending in `.pq`, builds the HTTP download URL, and records the local destination path for each file.

In [2]:
def fetch_xml(session, url):
    """Download and parse a THREDDS catalog XML document."""
    response = session.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return ET.fromstring(response.content)


def catalogue_refs(root):
    """Yield child catalog links, one for each site folder."""
    for ref in root.findall(".//thredds:catalogRef", NS):
        href = ref.attrib.get(XLINK)
        title = ref.attrib.get("{http://www.w3.org/1999/xlink}title") or ref.attrib.get("name")
        if href:
            yield title, href


def pq_datasets(root):
    """Yield dataset entries that point to pq files."""
    for dataset in root.findall(".//thredds:dataset", NS):
        name = dataset.attrib.get("name", "")
        url_path = dataset.attrib.get("urlPath")
        if url_path and name.endswith(".pq"):
            yield name, url_path


def local_path_from_url_path(url_path):
    """Map a THREDDS urlPath to the matching local data path."""
    marker = "fx31/publications/wind_nowcast/"
    if marker not in url_path:
        raise ValueError(f"Unexpected THREDDS urlPath: {url_path}")
    return DATA_DIR / url_path.split(marker, 1)[1]


def discover_pq_files(catalog_url=CATALOG_URL):
    """Return metadata for every pq file found in the site catalogs."""
    records = []
    with requests.Session() as session:
        root = fetch_xml(session, catalog_url)
        folder_refs = list(catalogue_refs(root))
        print(f"Found {len(folder_refs)} site folders:", ", ".join(name for name, _ in folder_refs))

        # Each folder has its own catalog.xml containing monthly csv and pq files.
        for folder_name, href in folder_refs:
            folder_catalog_url = urljoin(catalog_url, href)
            folder_root = fetch_xml(session, folder_catalog_url)
            for file_name, url_path in pq_datasets(folder_root):
                records.append({
                    "site": folder_name,
                    "name": file_name,
                    # Direct HTTP download endpoint for this THREDDS dataset.
                    "url": urljoin(THREDDS_ROOT, f"/thredds/fileServer/{url_path}"),
                    "local_path": local_path_from_url_path(url_path),
                })

    return records


pq_files = discover_pq_files()
print(f"Found {len(pq_files)} pq files")
pq_files[:5]

Found 8 site folders: bellambi, bonnie_rock, grove, kununurra, maryborough, strathalbyn, sydney_airport, warruwi
Found 112 pq files


[{'site': 'bellambi',
  'name': 'bellambi_202101.pq',
  'url': 'https://thredds.nci.org.au/thredds/fileServer/fx31/publications/wind_nowcast/bellambi/bellambi_202101.pq',
  'local_path': PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/bellambi/bellambi_202101.pq')},
 {'site': 'bellambi',
  'name': 'bellambi_202102.pq',
  'url': 'https://thredds.nci.org.au/thredds/fileServer/fx31/publications/wind_nowcast/bellambi/bellambi_202102.pq',
  'local_path': PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/bellambi/bellambi_202102.pq')},
 {'site': 'bellambi',
  'name': 'bellambi_202103.pq',
  'url': 'https://thredds.nci.org.au/thredds/fileServer/fx31/publications/wind_nowcast/bellambi/bellambi_202103.pq',
  'local_path': PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/bellambi/bellambi_202103.pq')},
 {'site': 'bellambi',
  'name': 'bellambi_202104.pq',
  'url': 'https://thredds.nci.org.au/thredds/fileServer/fx

## Download pq files

This section downloads the discovered `.pq` files. It is safe to re-run: files with the expected byte size are skipped, and in-progress downloads are written to `.part` files before being moved into place.

In [3]:
def remote_size(response):
    """Read content-length from the HTTP response when the server provides it."""
    value = response.headers.get("content-length")
    return int(value) if value and value.isdigit() else None


def download_one(session, record):
    """Download one pq file and return downloaded/skipped status."""
    local_path = record["local_path"]
    local_path.parent.mkdir(parents=True, exist_ok=True)

    with session.get(record["url"], stream=True, timeout=TIMEOUT) as response:
        response.raise_for_status()
        size = remote_size(response)

        # Skip files that are already present with the same remote byte size.
        if local_path.exists() and size is not None and local_path.stat().st_size == size:
            return "skipped"

        # Write to a temporary file first so interrupted downloads do not look complete.
        temp_path = local_path.with_suffix(local_path.suffix + ".part")
        progress = None
        if tqdm is not None:
            progress = tqdm(total=size, unit="B", unit_scale=True, desc=local_path.name, leave=False)

        try:
            with temp_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        file.write(chunk)
                        if progress is not None:
                            progress.update(len(chunk))
            # Atomic replace after a complete download.
            temp_path.replace(local_path)
        finally:
            if progress is not None:
                progress.close()

    return "downloaded"


def download_all(records):
    """Download all discovered files and report a small status summary."""
    counts = {"downloaded": 0, "skipped": 0, "failed": 0}
    failures = []

    iterator = tqdm(records, desc="pq files") if tqdm is not None else records
    with requests.Session() as session:
        for record in iterator:
            try:
                status = download_one(session, record)
                counts[status] += 1
            except Exception as exc:
                counts["failed"] += 1
                failures.append((record, exc))
                print(f"Failed: {record['url']} -> {exc}")
            time.sleep(0.05)

    print(counts)
    if failures:
        print("Failed files:")
        for record, exc in failures:
            print(record["url"], exc)
    return counts, failures


counts, failures = download_all(pq_files)

pq files: 100%|██████████| 112/112 [00:14<00:00,  7.58it/s]

{'downloaded': 0, 'skipped': 112, 'failed': 0}


## Merge pq files by site

Each site folder contains one `.pq` file per month. This section merges those monthly files into one file per site.

The merged files are written to `data/wind_nowcast/merged_by_site`, one `.pq` file per site folder. Keeping merged outputs in a separate folder avoids mixing derived files with the original downloaded files.

In [4]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "Merging pq/parquet files requires pyarrow. Install it first with: pip install pyarrow"
    ) from exc


# Derived files are kept separate from the raw downloaded site folders.
MERGED_DIR = DATA_DIR / "merged_by_site"
MERGED_DIR.mkdir(parents=True, exist_ok=True)


def site_source_dirs(data_dir=DATA_DIR):
    """Return raw site folders, excluding the merged output folder."""
    return sorted(
        path for path in data_dir.iterdir()
        if path.is_dir() and path.name != MERGED_DIR.name
    )


def merge_site_pq_files(site_dir, output_dir=MERGED_DIR):
    """Merge all monthly pq files in one site folder into one site-level pq file."""
    source_files = sorted(site_dir.glob("*.pq"))
    if not source_files:
        print(f"No pq files found in {site_dir}")
        return None

    output_path = output_dir / f"{site_dir.name}.pq"
    temp_path = output_path.with_suffix(output_path.suffix + ".part")

    writer = None
    rows = 0
    try:
        # Stream tables through ParquetWriter to preserve schema and avoid manual row handling.
        for file_path in source_files:
            table = pq.read_table(file_path)
            if writer is None:
                writer = pq.ParquetWriter(temp_path, table.schema)
            writer.write_table(table)
            rows += table.num_rows
    finally:
        if writer is not None:
            writer.close()

    # Replace the previous merged file only after the new one is fully written.
    temp_path.replace(output_path)
    print(f"Merged {len(source_files)} files -> {output_path} ({rows:,} rows)")
    return output_path


merged_files = []
for site_dir in site_source_dirs():
    merged_path = merge_site_pq_files(site_dir)
    if merged_path is not None:
        merged_files.append(merged_path)

merged_files

Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/bellambi.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/bonnie_rock.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/grove.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/kununurra.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/maryborough.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/strathalbyn.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/sydney_airport.pq (610,560 rows)
Merged 14 files -> /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_sit

[PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/bellambi.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/bonnie_rock.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/grove.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/kununurra.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/maryborough.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/strathalbyn.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/sydney_airport.pq'),
 PosixPath('/home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/warruwi.pq')]

## Preview bellambi merged file

Read `bellambi.pq` from the merged output folder and display the first 50 rows and last 50 rows. This is a quick sanity check that the merged file is readable and that the time index covers the expected start and end of the combined period.

In [5]:
import pandas as pd


# Load the merged Bellambi file produced in the previous section.
bellambi_path = DATA_DIR / "merged_by_site" / "bellambi.pq"
bellambi_df = pd.read_parquet(bellambi_path)

print(f"File: {bellambi_path}")
print(f"Shape: {bellambi_df.shape}")

print("\nFirst 50 rows")
display(bellambi_df.head(50))

print("\nLast 50 rows")
display(bellambi_df.tail(50))

File: /home/timekeeper/Documents/Development/BOM-Team/data/wind_nowcast/merged_by_site/bellambi.pq
Shape: (610560, 7)

First 50 rows


,STN_NUM,AIR_TEMP,DWPT,WND_SPD,WND_DIR,MAX_WND_GUST,mask
2021-01-01 00:00:00,68228,21.3,12.4,5.7,163.0,6.2,1.0
2021-01-01 00:01:00,68228,21.4,13.1,5.1,159.0,6.2,1.0
2021-01-01 00:02:00,68228,21.4,13.1,5.7,163.0,6.7,1.0
2021-01-01 00:03:00,68228,21.5,13.4,5.1,160.0,6.2,1.0
2021-01-01 00:04:00,68228,21.3,12.4,6.7,152.0,7.2,1.0
2021-01-01 00:05:00,68228,21.1,12.0,6.7,154.0,6.7,1.0
2021-01-01 00:06:00,68228,21.3,13.0,5.7,152.0,6.2,1.0
2021-01-01 00:07:00,68228,21.0,11.6,6.2,151.0,6.7,1.0
2021-01-01 00:08:00,68228,21.4,13.1,5.7,156.0,6.7,1.0
2021-01-01 00:09:00,68228,21.5,13.4,5.7,155.0,6.2,1.0



Last 50 rows


,STN_NUM,AIR_TEMP,DWPT,WND_SPD,WND_DIR,MAX_WND_GUST,mask
2022-02-28 23:10:00,68228,16.8,13.1,7.7,96.0,8.2,1.0
2022-02-28 23:11:00,68228,17.0,13.7,6.7,97.0,7.7,1.0
2022-02-28 23:12:00,68228,16.9,13.6,7.7,98.0,8.2,1.0
2022-02-28 23:13:00,68228,16.7,13.2,7.7,101.0,8.7,1.0
2022-02-28 23:14:00,68228,16.7,13.2,8.2,95.0,8.7,1.0
2022-02-28 23:15:00,68228,16.8,13.3,8.2,93.0,8.7,1.0
2022-02-28 23:16:00,68228,16.7,13.4,7.2,91.0,8.2,1.0
2022-02-28 23:17:00,68228,16.8,13.5,6.7,92.0,7.2,1.0
2022-02-28 23:18:00,68228,16.8,13.7,7.7,93.0,8.7,1.0
2022-02-28 23:19:00,68228,16.7,13.2,8.2,92.0,9.3,1.0
